# 2.4.1 Maxwell eigenvalue problem
We solve the Maxwell eigenvalue problem

$$
\int \operatorname{curl} u \, \operatorname{curl} v
= \lambda \int u v
$$

for $u, v \; \bot \; \nabla H^1$
using the PINVIT solver from the `ngsolve.solvers` module.

The orthogonality to gradient fields is important to eliminate the huge number of zero eigenvalues. The projection onto the orthogonal sub-space is implemented by a Poisson projection:

$$
P u = u - \nabla \Delta^{-1} \operatorname{div} u
$$

The algorithm and the example are taken from the PhD thesis of Sabine Zaglmayr, pp 145-150.

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

The domain is the Fichera cube. The eigenfunctions are singular at the re-entrant edges and at the re-entrant corner, where we want to apply geometric mesh refinement:

In [ ]:
cube1 = Box( (-1,-1,-1), (1,1,1) )

cube2 = Box( (0,0,0), (2,2,2) )
cube2.edges.hpref=1    # mark edges for geometric refinement

fichera = cube1-cube2

Draw (fichera);

In [ ]:
mesh = fichera.GenerateMesh(maxh=0.4)
mesh.RefineHP(levels=2, factor=0.2)
Draw (mesh);

We need the stiffness matrix and the mass matrix. The preconditioner is a sparse direct solver for the shifted problem:

In [ ]:
fes = HCurl(mesh, order=3)
print ("ndof =", fes.ndof)
u,v = fes.TnT()

a = BilinearForm(curl(u)*curl(v)*dx)
m = BilinearForm(u*v*dx)

apre = BilinearForm(curl(u)*curl(v)*dx + u*v*dx)
pre = preconditioners.Direct(apre, inverse="sparsecholesky")

The discrete gradient is a sparse matrix mapping the coefficients of an $H^1$ function to the coefficients of its gradient in $H(curl)$. With it we assemble the Laplace matrix for the projection, and combine the projected preconditioner:

In [ ]:
with TaskManager():
    a.Assemble()
    m.Assemble()
    apre.Assemble()

    # build gradient matrix as sparse matrix (and corresponding scalar FESpace)
    gradmat, fesh1 = fes.CreateGradient()

    gradmattrans = gradmat.CreateTranspose() # transpose sparse matrix
    math1 = gradmattrans @ m.mat @ gradmat   # multiply matrices
    math1[0,0] += 1     # fix the 1-dim kernel
    invh1 = math1.Inverse(inverse="sparsecholesky")

    # build the Poisson projector with operator algebra:
    proj = IdentityMatrix() - gradmat @ invh1 @ gradmattrans @ m.mat

    projpre = proj @ pre.mat

    evals, evecs = solvers.PINVIT(a.mat, m.mat, pre=projpre, num=12, maxit=20)

In [ ]:
print ("Eigenvalues")
for lam in evals:
    print (lam)

We copy the eigenvectors into a multi-dimensional `GridFunction` for visualization:

In [ ]:
gfu = GridFunction(fes, multidim=len(evecs))
for i in range(len(evecs)):
    gfu.vecs[i].data = evecs[i]
Draw (Norm(gfu), mesh, order=4, min=0, max=2);